# Essential & Vital Workers — Workflow Notebook

This notebook is the **guided walkthrough** for the essential-worker pipeline.

Use this notebook to:

1. Reproduce every CSV under `results/` from `data/` inputs.
2. See each pipeline stage step-by-step (not only the one-shot `run_pipeline`).
3. Compare the three **indoor-fraction** methods and understand when to use each.
4. Inspect overlap calibration, on-site housing adjustments, and ILO validation.

**Downstream:** `scripts/scale_up_processing.ipynb` and
`scripts/visualization/EssentialWorkers_Choropleth_Visualiser.ipynb`
read `results/EssentialWorkersByCountry.csv`.

### Inputs (`data/`)

| File | Role |
| --- | --- |
| `ISCO-08 OpinionPollCensus.xlsx` | In-house vital poll (0/1) at ISCO L4 |
| `Indoors_Environmentally_Controlled_data.csv` | O*NET indoor context % (controlled) |
| `Indoors_Not_Environmentally_Controlled.csv` | O*NET indoor context % (not controlled) |
| `ISCO_SOC_Crosswalk.csv` | SOC → ISCO-08 mapping |
| `ILO_ISCO_08_GLB.csv` | ILO employment by country × ISCO L2 |
| `LFData_WB_plus.xlsx` | World Bank labour force 2024 |
| `ILO_country_essential_workers_pct.xlsx` | ILO WESO 2023 published %essential |
| `job_exposure_matrix.xls` | *Optional* — JEM Location for `jem_location` method |

### Outputs (`results/`)

| File | Contents |
| --- | --- |
| `EssentialWorkersByCountry.csv` | Per-country counts & percentages |
| `EssentialWorkersByRegion.csv` | UN regional aggregates |
| `Essential_Workers_Validation.csv` | Model vs calibrated vs ILO %essential |
| `Group_Overlap_Calibration.csv` | Per-country × group overlap adjustments |
| `Onsite_Housing_Worker_Requirements.csv` | Housing-relevant totals (excl. ISCO 61+63) |
| `Indoor_Context_Sensitivity.csv` | Global totals under each indoor method (optional) |


## Methodology Overview

> ILO (2023). **World Employment and Social Outlook 2023: The value of essential work.**
> International Labour Organization.
> [WCMS_871016](https://www.ilo.org/sites/default/files/wcmsp5/groups/public/@dgreports/@dcomm/@publ/documents/publication/wcms_871016.pdf)

The ILO classifies a worker as a *key worker* (essential worker) if **both** of these are true:

1. They are in a key **occupation** (ISCO-08 code listed in Table A2 of the report).
2. They are in a key **industry** (ISIC Rev.4 code listed in Table A1 of the report).

So `essential = occupation AND industry`, not just one or the other. The ILO computes its per-country shares from worker-level microdata in which every respondent has both an ISCO code and an ISIC code, so the intersection is exact.

### Our calculations

**We do not have access to ILO worker-level ISCO x ISIC microdata.** The only country-level breakdown we have is ILO employment per ISCO-08 L2 code (`ILO_ISCO_08_GLB.csv`). To approximate the intersection we use Figure A1 from the linked report above as **global priors** (`GROUP_OVERLAP[g]`) and then **calibrate** them per country.

**1. Global priors.** For each occupational group *g*, `GROUP_OVERLAP[g]` is the globally aggregated fraction of workers in group *g* that are also in a key ISIC industry (ILO Figure A1). Armed Forces uses 0.40 (Blueprint; ILO excludes uniformed services from headline figures).

The global average group overlap factors:

| Group | Overlap | Source |
| --- | ---: | --- |
| Food | 0.895 | ILO Figure A1 |
| Health | 0.819 | ILO Figure A1 |
| Retail | 0.876 | ILO Figure A1 |
| Security | 0.846 | ILO Figure A1 |
| Transport | 0.869 | ILO Figure A1 |
| Manual | 0.335 | ILO Figure A1 |
| Cleaning | 0.485 | ILO Figure A1 |
| Tech | 0.320 | ILO Figure A1 |
| Armed Forces | **0.40** | Blueprint Biosecurity's "A theory of pandemic-proof PPE" https://blueprintbiosecurity.org/u/2024/05/BB_Next-Gen-Report_PRF9-WEB-1.pdf?utm_source=bluedot-impact (as the ILO excludes armed forces from its global figures) |

**2. Per-country calibration.** For each country with ILO ISCO employment and a published WESO %essential, one scalar `x ∈ [0, 1]` moves all eight calibratable groups together: toward 1.0 when the model under-shoots ILO, toward 0 when it over-shoots. Armed Forces overlap stays fixed at 0.40. This is implemented in `essential_workers.calibrate_group_overlaps` and logged in `results/Group_Overlap_Calibration.csv`. For countries without data (e.g. China), calibrated overlaps are the mean of `SIMILAR_ISO3` neighbours' calibrated values.

**3. Applying overlaps to calculate vital workers** using the calibrated overlaps obtained from step 2, we apply them to a smaller set of ISCO-08 job categories. These were defined by team members each flagging ISCO-08 job categories as vital or not vital at the 4-digit code level (specific job categories). As the ILO employment data is at the 2-digit level (broader job categories), we take the mean of the 4-digit codes within each 2-digit category.


**4. Estimate indoor workers** We multiply the essential worker weights and vital worker weights by `indoors_context` (0–1), which is the fraction of that job category spent inside
and therefore benefitting from in-room air filtration. We derive this from O*NET or JEM datasets on time spent working indoors by job category.


## 0. Setup


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print  # running as plain script

REPO = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

import essential_workers as ew

DATA = REPO / 'data/essential_workers'
RESULTS = REPO / 'results/essential_workers'
RESULTS.mkdir(exist_ok=True)

JEM_PATH = DATA / 'job_exposure_matrix.xls'
HAS_JEM = JEM_PATH.exists()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

print(f'Repo:    {REPO}')
print(f'Data:    {DATA}  (exists={DATA.is_dir()})')
print(f'Results: {RESULTS}')
print(f'JEM file: {JEM_PATH.name} present={HAS_JEM}')


Repo:    /Users/james/Documents/Github/InRoomAirFilterScaleUp
Data:    /Users/james/Documents/Github/InRoomAirFilterScaleUp/data/essential_workers  (exists=True)
Results: /Users/james/Documents/Github/InRoomAirFilterScaleUp/results/essential_workers
JEM file: job_exposure_matrix.xls present=True


## 1. Quick-start — full pipeline

One call mirrors `python -m essential_workers` and writes all standard CSVs.
Set `write_indoor_sensitivity=True` to also save `Indoor_Context_Sensitivity.csv`.


In [2]:
outputs = ew.run_pipeline(
    data_dir=DATA,
    results_dir=RESULTS,
    write=True,
    write_indoor_sensitivity=True,
)
lf = outputs.labour_force_df
v = outputs.validation
vm = outputs.validation_model
cal_detail = outputs.overlap_calibration.detail_df

global_summary = ew.compute_global_worker_summary(lf)
print(f"Global labour force: {global_summary.attrs['labour_force']:.3e}")
display(global_summary)

range_compression = ew.summarize_indoor_range_compression(lf)
print(
    "\nCountry-level share range: total vs indoor "
    "(negative % change ⇒ indoor shares more similar across countries).\n"
    "Absolute spread: Pitman–Morgan on raw % (p indoor SD < total).\n"
    "Relative spread: CV = SD/mean, and Pitman–Morgan on log % "
    "(p indoor log-SD < total)."
)
display(range_compression)

group_range = ew.summarize_group_indoor_range_compression(outputs.group_df, lf)
print(
    "\nSame range compression by occupational group "
    "(ILO-employment countries only; sorted by largest Essential range reduction)"
)
display(group_range)

rankings = ew.rank_countries_by_worker_pct(lf, n=10)
for label, (top, bottom) in rankings.items():
    print(f"\nTop 10 countries by {label}")
    display(top)
    print(f"Bottom 10 countries by {label}")
    display(bottom)

print(f"\nCalibrated global %Essential: {v.global_pct_essential:.2f}%")
print(f"Model (global overlap) |Δ| mean: {vm.mean_abs_delta_pp:.2f} pp")
print(f"Calibrated           |Δ| mean: {v.mean_abs_delta_pp:.2f} pp   r={v.correlation:.3f}")

housing = outputs.onsite_housing_df
g = housing.loc[housing['Country Code'] == 'GLOBAL'].iloc[0]
print(f"\nOn-site housing (excl. ISCO 61+63): Essential {g['Essential Workers (Housing Requirement)']:.3e}")


Global labour force: 3.732e+09


,Workers,% of Labour Force,Country min %,Country max %
Category,,,,
Essential workers,1.980536e+09,53.074240,19.574882,87.000720
Indoor essential workers,1.080393e+09,28.952288,6.831966,48.789163
Outdoor essential workers,9.001426e+08,24.121952,5.979428,76.537416
Vital workers,1.268512e+09,33.993500,9.062232,78.061091
Indoor vital workers,5.439453e+08,14.576605,2.740385,27.859115
Outdoor vital workers,7.245671e+08,19.416895,3.426795,75.320706



Country-level share range: total vs indoor (negative % change ⇒ indoor shares more similar across countries).
Absolute spread: Pitman–Morgan on raw % (p indoor SD < total).
Relative spread: CV = SD/mean, and Pitman–Morgan on log % (p indoor log-SD < total).


,n countries,Total country min %,Total country max %,Total range (pp),Total SD (pp),Total mean %,Total CV,Indoor country min %,Indoor country max %,Indoor range (pp),Indoor SD (pp),Indoor mean %,Indoor CV,Δ range (pp),% change in range,Variance ratio (indoor/total),CV ratio (indoor/total),Pitman–Morgan p (two-sided),p (indoor SD < total),p (indoor log-SD < total)
Transition,,,,,,,,,,,,,,,,,,,,
Essential → Indoor essential,216,19.574882,87.000720,67.425838,15.850583,49.417811,0.320746,6.831966,48.789163,41.957197,5.905741,26.484042,0.222992,-25.468641,-37.772821,0.138822,0.69523,2.681028e-93,1.340514e-93,7.793349e-09
Vital → Indoor vital,216,9.062232,78.061091,68.998859,15.702034,29.992873,0.523526,2.740385,27.859115,25.118729,4.051091,11.932855,0.339490,-43.880130,-63.595442,0.066563,0.64847,5.544462e-241,2.772231e-241,3.081364e-15



Same range compression by occupational group (ILO-employment countries only; sorted by largest Essential range reduction)


...


Top 10 countries by % Indoor vital


,Country Name,Country Code,% Indoor vital
0,Tanzania,TZA,27.859115
1,Côte d'Ivoire,CIV,25.477921
2,Bangladesh,BGD,23.504060
3,Bhutan,BTN,22.310974
4,St. Kitts and Nevis,KNA,22.173410
5,St. Vincent and the Grenadines,VCT,22.173410
6,Dominica,DMA,22.173410
7,St. Lucia,LCA,22.173410
8,Madagascar,MDG,22.173410
9,Saint-Martin,MAF,22.173410


Bottom 10 countries by % Indoor vital


,Country Name,Country Code,% Indoor vital
0,Burundi,BDI,2.740385
1,Mozambique,MOZ,4.640643
2,Burkina Faso,BFA,4.773030
3,Singapore,SGP,5.591291
4,Luxembourg,LUX,5.635436
5,Lesotho,LSO,6.632930
6,Cyprus,CYP,6.718901
7,Monaco,MCO,6.718901
8,San Marino,SMR,6.718901
9,Liechtenstein,LIE,6.718901



Calibrated global %Essential: 53.07%
Model (global overlap) |Δ| mean: 4.69 pp
Calibrated           |Δ| mean: 0.90 pp   r=0.982

On-site housing (excl. ISCO 61+63): Essential 1.316e+09


### Occupational-group composition

Worker-weighted breakdown of who makes up the essential / vital workforces
(Food, Health, Retail, …). Global / region / country CSVs are written by
`run_pipeline`; the figure is from `plot_group_composition.py`.


In [ ]:
group_comp = ew.summarize_group_composition(outputs.group_df)
display(
    group_comp.assign(
        **{
            c: (group_comp[c] * 100).round(1)
            for c in ew.GROUP_COMPOSITION_SHARE_COLS
        }
    )[["occupational_group", *ew.GROUP_COMPOSITION_SHARE_COLS]]
)
print(
    "\nAlso available: "
    "summarize_group_composition(..., by='Region'|'Country') "
    "→ EssentialWorkersByGroupComposition_*.csv"
)


## 2. Preprocessing — load inputs & build ISCO weights

This section mirrors the **first half** of `run_pipeline`: read files, build the L2
weight table (poll + indoor context + ILO essential flag + group overlaps).

### 2.1 Load raw inputs


In [3]:
poll_df = pd.read_excel(DATA / 'ISCO-08 OpinionPollCensus.xlsx', engine='openpyxl')
onet_env_df = pd.read_csv(DATA / 'Indoors_Environmentally_Controlled_data.csv')
onet_not_df = pd.read_csv(DATA / 'Indoors_Not_Environmentally_Controlled.csv')
crosswalk_df = pd.read_csv(DATA / 'ISCO_SOC_Crosswalk.csv')
ilo_emp_df = pd.read_csv(DATA / 'ILO_ISCO_08_GLB.csv')
lf_raw = pd.read_excel(DATA / 'LFData_WB_plus.xlsx', usecols=[0, 1, 3])
ilo_pct_df = ew.load_ilo_published_pct(DATA / 'ILO_country_essential_workers_pct.xlsx')

for name, df in [
    ('poll', poll_df), ('ONET env', onet_env_df), ('ONET not', onet_not_df),
    ('crosswalk', crosswalk_df), ('ILO emp', ilo_emp_df), ('LF', lf_raw), ('ILO pct', ilo_pct_df),
]:
    print(f'{name:12s} {df.shape}')


poll         (436, 39)
ONET env     (879, 4)
ONET not     (894, 4)
crosswalk    (1125, 6)
ILO emp      (38734, 11)
LF           (216, 3)
ILO pct      (90, 3)


### 2.2 Weight template (before group overlaps)

`_build_isco_lvl2_template` attaches `indoors_context` at L4, averages to L2, sets
`Essential Weight ILO`, maps `Group`, and applies poll/ILO teleworkable rules.
Default indoor method here is `onet_max` (same as `run_pipeline`).


In [4]:
weights_template = ew._build_isco_lvl2_template(
    poll_df,
    crosswalk_df,
    onet_controlled_df=onet_env_df,
    onet_not_controlled_df=onet_not_df,
    indoor_context_method='onet_max',
    jem_path=JEM_PATH if HAS_JEM else None,
)
weights = ew.apply_group_overlaps(weights_template, ew.GROUP_OVERLAP)

cols = [
    'Vital Weight POLL', ew.INDOORS_CONTEXT_COLUMN, 'Essential Weight ILO',
    'Group', 'Group Overlap',
    'ISCO_08_PollWeights', 'ISCO_08_ILOWeights',
    'ISCO_08_PollWeights_Total', 'ISCO_08_ILOWeights_Total',
]
display(weights[cols].head(12))

print('Subsistence farmers (63) indoor context:', weights.at['63', ew.INDOORS_CONTEXT_COLUMN])
print('NON_ILO poll-zero codes:', ew.NON_ILO_POLL_CODES)


,Vital Weight POLL,indoors_context,Essential Weight ILO,Group,Group Overlap,ISCO_08_PollWeights,ISCO_08_ILOWeights,ISCO_08_PollWeights_Total,ISCO_08_ILOWeights_Total
ISCO-08,,,,,,,,,
01,0.400000,1.000000,1,ArmedForces,0.400,0.16000,0.400000,0.1600,0.400
02,0.400000,1.000000,1,ArmedForces,0.400,0.16000,0.400000,0.1600,0.400
03,0.400000,1.000000,1,ArmedForces,0.400,0.16000,0.400000,0.1600,0.400
11,0.000000,0.838000,0,NaN,0.000,0.00000,0.000000,0.0000,0.000
12,0.000000,0.866136,0,NaN,0.000,0.00000,0.000000,0.0000,0.000
13,0.000000,0.853611,0,NaN,0.000,0.00000,0.000000,0.0000,0.000
14,0.000000,0.861111,0,NaN,0.000,0.00000,0.000000,0.0000,0.000
21,0.000000,0.854463,0,NaN,0.000,0.00000,0.000000,0.0000,0.000
22,0.533333,0.910921,1,Health,0.819,0.39789,0.746044,0.4368,0.819


Subsistence farmers (63) indoor context: 0.0
NON_ILO poll-zero codes: ['13', '21', '33', '35']


### 2.3 Per-country ILO employment by ISCO L2

`build_employment_by_isco` builds one employment snapshot per country:

- **Sex:** only `sex.label == "Total"` (Male/Female rows are dropped).
- **Year:** one survey year for all ISCO codes in that country — the latest year with NEC (“Not elsewhere classified”) ≤ 10% of total employment; if every year is above 10%, the year with the **lowest** NEC share (ties → more recent year).
- **Headcount:** `obs_value` (thousands) × 1000; ISCO L2 keys are stripped/normalised (`22`, `Tot`, `Not`).

In `compute_worker_dicts`, NEC employment is added to essential/vital numerators at each country’s **employment-weighted average** essential/vital weight over coded occupations (excluding `Tot` and `Not`). The official `Tot` row remains the denominator for % columns.


In [5]:
employment_by_iso = ew.build_employment_by_isco(ilo_emp_df)
sample = next(iter(employment_by_iso))
print(f'Countries with ILO breakdown: {len(employment_by_iso)}')
display(pd.Series(employment_by_iso[sample]).head(12))


Countries with ILO breakdown: 149


Tot    7679474.0
02      209728.0
11        7876.0
12       89945.0
13       23102.0
14       10688.0
21       22794.0
22       63733.0
23      236394.0
24        9108.0
26       58692.0
32        9930.0
dtype: float64

## 3. Indoor-fraction methods — comparison

The **indoor fraction** (`indoors_context`, 0–1) scales indoor vital/essential counts.
It does **not** change total vital/essential (the `_Total` weight columns omit it).

| Method | Source | Rule | When to use |
| --- | --- | --- | --- |
| **`onet_max`** (default) | Both O*NET CSVs | Per SOC: `max(env, not_env)` context % ÷ 100 | Main results; smooth 0–1; matches original notebook intent |
| **`onet_banded`** | Same O*NET | ≥75% → 1.0; 50–75% → 0.5; else 0 | Conservative / step-function sensitivity; reduces partial-indoor occupations |
| **`jem_location`** | `job_exposure_matrix.xls` | Mean JEM Location per ISCO L4 → bucket {0, 0.5, 1} | Alternative occupational exposure database; requires JEM file |

All three methods still use the same poll, ILO essential flags, and overlap calibration.


### 3.1 L2 `indoors_context` under each method


In [6]:
def weights_for_method(method: ew.IndoorContextMethod) -> pd.DataFrame:
    return ew.build_isco_lvl2_weights(
        poll_df, crosswalk_df,
        onet_controlled_df=onet_env_df,
        onet_not_controlled_df=onet_not_df,
        indoor_context_method=method,
        jem_path=JEM_PATH if method == 'jem_location' else None,
    )

methods: list[ew.IndoorContextMethod] = ['onet_max', 'onet_banded']
if HAS_JEM:
    methods.append('jem_location')
else:
    print('Skipping jem_location — job_exposure_matrix.xls not in data/')

indoor_compare = pd.DataFrame({
    m: weights_for_method(m)[ew.INDOORS_CONTEXT_COLUMN] for m in methods
})
indoor_compare['spread'] = indoor_compare.max(axis=1) - indoor_compare.min(axis=1)
print('L2 codes where methods disagree most (top 10 by spread):')
display(indoor_compare.sort_values('spread', ascending=False).head(10))
print('\nSummary stats of indoors_context by method:')
display(indoor_compare[methods].describe().T)


L2 codes where methods disagree most (top 10 by spread):


,onet_max,onet_banded,jem_location,spread
ISCO-08,,,,
12,0.866136,0.964286,0.000000,0.964286
25,0.849585,0.918985,0.000000,0.918985
24,0.828455,0.912121,0.000000,0.912121
43,0.773333,0.750000,0.083333,0.690000
44,0.877500,0.900000,0.214286,0.685714
35,0.890000,1.000000,0.333333,0.666667
21,0.854463,0.946037,0.333333,0.612704
33,0.831627,0.886586,0.291667,0.594919
62,0.552667,0.500000,0.000000,0.552667



Summary stats of indoors_context by method:


,count,mean,std,min,25%,50%,75%,max
onet_max,43.0,0.756783,0.180103,0.0,0.711640,0.801136,0.854037,1.0
onet_banded,43.0,0.751107,0.261960,0.0,0.684544,0.816667,0.938378,1.0
jem_location,43.0,0.583184,0.371500,0.0,0.302083,0.666667,0.905882,1.0


### 3.2 O*NET SOC → ISCO example (how `onet_max` vs `onet_banded` differ)

At SOC level we take the max of both context columns, then map through the crosswalk.


In [7]:
soc_merged = ew._merge_onet_max_context(onet_env_df, onet_not_df)
sample_soc = soc_merged.head(5).copy()
sample_soc['onet_max_frac'] = sample_soc['context_pct'].apply(
    lambda p: ew._pct_to_indoor_fraction(p, 'onet_max')
)
sample_soc['onet_banded_frac'] = sample_soc['context_pct'].apply(
    lambda p: ew._pct_to_indoor_fraction(p, 'onet_banded')
)
display(sample_soc)


,Code,context_pct,onet_max_frac,onet_banded_frac
0,11-1011,96.0,0.96,1.0
1,11-1011,96.0,0.96,1.0
2,11-1011,94.0,0.94,1.0
3,11-1011,94.0,0.94,1.0
4,11-1021,85.0,0.85,1.0


### 3.3 Global worker totals — side-by-side


In [8]:
sensitivity = ew.compare_indoor_context_methods(DATA)
pivot = sensitivity.pivot(
    index='Category', columns='indoor_context_method', values='% of Labour Force'
)
display(pivot.round(2))

if (RESULTS / 'Indoor_Context_Sensitivity.csv').exists():
    print(f'Also on disk: {RESULTS / "Indoor_Context_Sensitivity.csv"}')


indoor_context_method,jem_location,onet_banded,onet_max
Category,,,
Essential workers,53.07,53.07,53.07
Indoor essential workers,22.36,28.95,31.81
Indoor vital workers,8.44,14.58,17.23
Outdoor essential workers,30.71,24.12,21.26
Outdoor vital workers,25.56,19.42,16.77
Vital workers,33.99,33.99,33.99


Also on disk: /Users/james/Documents/Github/InRoomAirFilterScaleUp/results/essential_workers/Indoor_Context_Sensitivity.csv


## 4. Group overlap calibration

Before attaching to the labour-force table, the pipeline:

1. Computes **model** workers with global `GROUP_OVERLAP`.
2. Solves per-country scalar `x` so essential mass matches ILO published % (where data exist).
3. **Back-fills** missing countries from `SIMILAR_ISO3` neighbours, else global fallback.

Vital and essential **totals** both use calibrated overlaps; only indoor counts use `indoors_context`.


In [9]:
lf_prep = ew.prepare_labour_force(lf_raw)
lf_prep = ew.fill_missing_labour_force_from_ilo_tot(lf_prep, employment_by_iso)
workers_model = ew.compute_worker_dicts(employment_by_iso, weights_template)

overlap_result = ew.calibrate_country_overlaps(
    lf_prep, employment_by_iso, ilo_pct_df, weights_template,
    workers_model=workers_model,
)
overlap_country = overlap_result.country_table
workers_cal = ew.compute_worker_dicts(
    employment_by_iso, weights_template, overlap_result.overlaps_by_country,
)

print('Overlap sources:', overlap_country['overlap_source'].value_counts().to_dict())
print('\nSample calibrated country (first ILO-calibrated row):')
ilo_rows = overlap_country[overlap_country['overlap_source'] == ew.OVERLAP_SOURCE_ILO]
display(ilo_rows.head(3)[[c for c in ilo_rows.columns if 'overlap_' in c or c.startswith('calibration')]])


Overlap sources: {'ilo_calibrated': 86, 'global_fallback': 67, 'neighbour_backfill': 45, '': 18}

Sample calibrated country (first ILO-calibrated row):


,overlap_Food,overlap_Health,overlap_Retail,overlap_Security,overlap_Transport,overlap_Manual,overlap_Cleaning,overlap_Tech,calibration_x,calibration_direction,overlap_source
1,0.872799,0.798684,0.854270,0.825015,0.847444,0.326690,0.472969,0.312062,0.024805,lower,ilo_calibrated
2,0.923239,0.867678,0.909349,0.887417,0.904231,0.513845,0.623504,0.502879,0.268940,raise,ilo_calibrated
3,0.874292,0.800051,0.855732,0.826426,0.848894,0.327249,0.473779,0.312596,0.023137,lower,ilo_calibrated


In [10]:
# Rebuild calibration detail (same as run_pipeline after worker dicts exist)
cal_detail = ew.build_group_overlap_calibration_detail(
    lf_prep, overlap_country, employment_by_iso, weights_template, ilo_pct_df,
    workers_model, workers_cal,
)
print('Largest |adjustment| by group (ILO-calibrated countries):')
ilo_cal = cal_detail[cal_detail['Overlap source'] == ew.OVERLAP_SOURCE_ILO]
display(
    ilo_cal.groupby('Group')['Adjustment']
    .apply(lambda s: s.abs().mean())
    .sort_values(ascending=False)
    .to_frame('mean |adjustment|')
)


Largest |adjustment| by group (ILO-calibrated countries):


,mean |adjustment|
Group,
Tech,0.087779
Manual,0.086789
Cleaning,0.076892
Health,0.054853
Security,0.053072
Transport,0.051554
Retail,0.051092
Food,0.049839
ArmedForces,0.000000


## 5. Labour force join, back-fill, and absolute counts

Matches `run_pipeline` after calibration: attach % columns, neighbour back-fill,
on-site excluded shares, then multiply by labour force.

Where World Bank `Labour Force (2024)` is missing (e.g. Palestine), the pipeline
uses ILO `Tot` employment from the selected country-year snapshot (persons).


In [11]:
lf = ew.attach_pct_columns(lf_prep.copy(), workers_cal)
print('NaN %Essential before neighbour back-fill:', lf['%Essential Workers'].isna().sum())
lf = ew.backfill_neighbours(lf)
print('NaN %Essential after  neighbour back-fill:', lf['%Essential Workers'].isna().sum())

lf = ew.attach_onsite_excluded_pct(
    lf, employment_by_iso, weights_template, overlap_result.overlaps_by_country,
)
lf = ew.backfill_neighbours(
    lf, cols=[ew.ONSITE_EXCLUDED_ESSENTIAL_PCT_COL, ew.ONSITE_EXCLUDED_VITAL_PCT_COL],
)
lf = ew.compute_absolute_counts(lf)
regional = ew.aggregate_by_region(lf)

display(lf[['Country Name', '%Indoor Essential Workers', '%Essential Workers',
              ew.ONSITE_EXCLUDED_ESSENTIAL_PCT_COL]].head(8))


NaN %Essential before neighbour back-fill: 72
NaN %Essential after  neighbour back-fill: 0


KeyError: "Columns not found: 'Indoor Essential CADR Requirement (L/s)', 'Indoor Vital CADR Requirement (L/s)'"

## 6. On-site housing worker requirements

ISCO **61** (market-oriented skilled agricultural) and **63** (subsistence farmers)
are treated as already on-site; they are subtracted from housing-relevant totals
(weighted by each series' total-weight column).


In [ ]:
onsite_df = ew.build_onsite_housing_worker_requirements(lf)
global_row = onsite_df.loc[onsite_df['Country Code'] == 'GLOBAL'].iloc[0]
print('Global housing-relevant workers:')
for col in ew.ONSITE_HOUSING_WORKER_COUNT_COLUMNS:
    print(f'  {col}: {global_row[col]:.3e}')
display(onsite_df.head(10))


## 7. Validation — model vs calibrated vs ILO

Dual validation is written to `Essential_Workers_Validation.csv` when `write=True`.


In [ ]:
val_cal = ew.validate_against_ilo(lf, ilo_pct_df, our_pct_label='Our %Essential (calibrated)')
lf_model = ew.attach_pct_columns(lf.copy(), workers_model)
val_model = ew.validate_against_ilo(
    lf_model, ilo_pct_df, our_pct_label='Our %Essential (model, global overlap)',
)
merged_val = ew.build_dual_validation_merged(lf, ilo_pct_df, workers_model, val_cal)

print(f'Global calibrated %Essential: {val_cal.global_pct_essential:.2f}%')
print(f'Mean |Δ| calibrated: {val_cal.mean_abs_delta_pp:.2f} pp   model: {val_model.mean_abs_delta_pp:.2f} pp')
print(f'Outliers >{val_cal.outlier_threshold_pp:.0f} pp: {len(val_cal.outlier_df)}')
display(val_cal.outlier_df[[
    'Country Name', 'Our %Essential (pct)', 'ILO %essential (published)', 'Delta (pp)',
]].head(10))


### 7.1 Regional aggregates


In [ ]:
display(regional[[
    'Region', 'Labour Force (2024)', 'Indoor Essential Workers',
    'Essential Workers', '%Indoor Essential Workers',
]].head(12))


## 8. Optional visualization

The manuscript figures come from `scripts/visualization/`, which reads
`results/essential_workers/EssentialWorkersByCountry.csv`:

```bash
python scripts/visualization/plot_essential_workers.py  # worker share maps
python scripts/visualization/plot_workers_vs_gdp.py     # worker shares against GDP
python scripts/visualization/plot_group_composition.py  # occupational composition
```

Quick static view of the top countries by % indoor essential:


In [ ]:
try:
    import matplotlib.pyplot as plt
    top = lf.nlargest(15, '%Indoor Essential Workers')
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(top['Country Name'], top['%Indoor Essential Workers'] * 100)
    ax.set_xlabel('% Indoor Essential Workers')
    ax.set_title('Top 15 countries — % indoor essential (calibrated pipeline)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skip bar chart or pip install matplotlib')


## 9. Write outputs & re-run tests

The quick-start cell already wrote CSVs. To refresh without re-running earlier cells:

```python
ew.run_pipeline(DATA, RESULTS, write=True, write_indoor_sensitivity=True)
```

Tests (from repo root):

```bash
pytest tests/test_essential_workers.py
pytest tests/test_essential_workers.py --full-data  # uses data/ not fixtures
```


In [ ]:
# Uncomment to persist again after editing constants in essential_workers.py:
# ew.run_pipeline(DATA, RESULTS, write=True, write_indoor_sensitivity=True)
print('Pipeline outputs expected in:', RESULTS)
for f in sorted(RESULTS.glob('Essential*.csv')) + sorted(RESULTS.glob('*Overlap*.csv')) + sorted(RESULTS.glob('Onsite*.csv')) + sorted(RESULTS.glob('Indoor_Context*.csv')):
    print(' ', f.name, f.stat().st_size if f.exists() else 'missing')
